In [1]:
%pip -q install alpaca-py


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pytz


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from alpaca.trading.client import TradingClient
from alpaca.data.historical import StockHistoricalDataClient

trading = TradingClient(
    os.getenv("ALPACA_API_KEY"),
    os.getenv("ALPACA_API_SECRET"),
    paper=True
)

data = StockHistoricalDataClient(
    os.getenv("ALPACA_API_KEY"),
    os.getenv("ALPACA_API_SECRET")
)

acct = trading.get_account()
print("✅ Connected. Cash:", acct.cash, "Buying power:", acct.buying_power)


✅ Connected. Cash: 100000 Buying power: 200000


In [4]:
from alpaca.data.requests import StockLatestTradeRequest
from alpaca.data.enums import DataFeed

def get_latest_price(symbol: str) -> float:
    req = StockLatestTradeRequest(
        symbol_or_symbols=symbol,
        feed=DataFeed.IEX   # 👈 THIS is the fix
    )
    latest = data.get_stock_latest_trade(req)
    return float(latest[symbol].price)


In [5]:
PAPER = True
SYMBOLS = ["AAPL", "MSFT", "NVDA"]

MAX_USD_PER_TRADE = 200      # hard cap per order
MAX_OPEN_POSITIONS = 3       # hard cap positions count
ALLOW_SHORTS = False         # keep simple


In [6]:
def get_positions_by_symbol() -> dict:
    positions = trading.get_all_positions()
    return {p.symbol: p for p in positions}

def can_open_new_position() -> bool:
    pos = get_positions_by_symbol()
    return len(pos) < MAX_OPEN_POSITIONS

print("Positions:", list(get_positions_by_symbol().keys()))
print("Can open new?", can_open_new_position())


Positions: []
Can open new? True


In [7]:
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import DataFeed
from datetime import datetime, timedelta, timezone

def get_sma(symbol: str, days: int = 20) -> float:
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days * 3)

    req = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=TimeFrame.Day,
        start=start,
        end=end,
        limit=days,
        feed=DataFeed.IEX
    )

    bars = data.get_stock_bars(req).data.get(symbol, [])
    closes = [float(b.close) for b in bars][-days:]

    if len(closes) < max(5, days // 2):
        raise RuntimeError(f"Not enough DAILY bars for {symbol}: got {len(closes)}")

    return sum(closes) / len(closes)


In [8]:
def decide(symbol: str) -> str:
    price = get_latest_price(symbol)
    sma = get_sma(symbol, days=20)
    holding = symbol in get_positions_by_symbol()

    if price > sma and not holding:
        return "BUY"
    if price < sma and holding:
        return "SELL"
    return "HOLD"


In [9]:
SYMBOLS = ["AAPL", "MSFT", "NVDA"]


In [10]:
for s in SYMBOLS:
    try:
        print(s, decide(s))
    except Exception as e:
        print(s, "ERROR:", e)


AAPL HOLD
MSFT HOLD
NVDA HOLD


In [11]:
MAX_DAILY_LOSS_USD = 300   # bot shuts down if it loses this much in a day

def daily_pnl() -> float:
    acct = trading.get_account()
    return float(acct.equity) - float(acct.last_equity)

def kill_switch_triggered() -> bool:
    pnl = daily_pnl()
    print("Daily PnL:", pnl)
    return pnl <= -MAX_DAILY_LOSS_USD


In [12]:
def already_holding(symbol: str) -> bool:
    return symbol in get_positions_by_symbol()


In [13]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

def place_market_order(symbol: str, side: str, max_usd: float):
    price = get_latest_price(symbol)
    qty = int(max_usd // price)  # whole shares

    if qty <= 0:
        print(f"Skip {symbol}: price {price:.2f} too high for ${max_usd}")
        return None

    order = trading.submit_order(MarketOrderRequest(
        symbol=symbol,
        qty=qty,
        side=OrderSide.BUY if side == "BUY" else OrderSide.SELL,
        time_in_force=TimeInForce.DAY
    ))

    print(f"✅ Submitted {side} {symbol} qty={qty} (approx ${qty*price:.2f})")
    return order


In [14]:
def run_once():
    print("\n=== BOT TICK ===")

    if kill_switch_triggered():
        print("⛔ Kill switch triggered. No trades.")
        return

    positions = get_positions_by_symbol()

    for symbol in SYMBOLS:
        try:
            action = decide(symbol)
            print(symbol, "→", action)

            if action == "BUY":
                if already_holding(symbol):
                    print("Already holding", symbol)
                    continue
                if len(positions) >= MAX_OPEN_POSITIONS:
                    print("Max positions reached")
                    continue
                place_market_order(symbol, "BUY", MAX_USD_PER_TRADE)

            elif action == "SELL" and symbol in positions:
                qty = int(float(positions[symbol].qty))
                trading.submit_order(
                    MarketOrderRequest(
                        symbol=symbol,
                        qty=qty,
                        side=OrderSide.SELL,
                        time_in_force=TimeInForce.DAY
                    )
                )
                print("Sold", symbol, qty)

        except Exception as e:
            print(symbol, "ERROR:", e)
